# PPO + World Model Colab Demo

**Fastest path (no GPU needed):** Run cells 1-4. These load pre-committed artifacts (PPO benchmark JSON, world-model reward-faithfulness table) and evaluate the tracked checkpoint without any training.

**Full smoke run:** Cells 5-9 collect a tiny replay dataset, train a 1-epoch RSSM, and generate a hallucination video. These work on CPU (~10 min) or GPU (~2 min). Switch the runtime to GPU for speed: Runtime -> Change runtime type -> T4.

What this notebook does *not* do:
- Reproduce the full P5 world-model run (requires A100 + hours)
- Reproduce the multi-agent Prime Intellect training

In [ ]:
# Setup
!git clone https://github.com/yuvimalik/Racing_Gym_RL.git 2>/dev/null || (cd Racing_Gym_RL && git pull origin main)
%cd Racing_Gym_RL

!apt-get update -qq
!apt-get install -y -qq xvfb swig python3-opengl ffmpeg > /dev/null
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q -r requirements_sb3.txt --no-deps
!pip install -q git+https://github.com/igilitschenski/multi_car_racing.git --no-deps

import json
import os
from pathlib import Path

import torch

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("Running on CPU. Cells 1-4 work fine; smoke-run cells take ~10 min.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# PPO baseline summary from tracked autoresearch artifacts
import pandas as pd

best_metrics = json.loads(Path("autoresearch/results/best_metrics.json").read_text())
ppo_summary = pd.DataFrame([
    {
        "run": "autoresearch_best",
        "mean_reward": best_metrics["mean_reward"],
        "mean_progress": best_metrics["mean_progress"],
        "mean_speed": best_metrics["mean_speed"],
        "offtrack_rate": best_metrics["offtrack_rate"],
        "steps_per_second": best_metrics["steps_per_second"],
    }
])
ppo_summary

In [ ]:
# Evaluate the tracked PPO checkpoint
!xvfb-run -a python evaluate.py --model models/best_model_torch.pt --episodes 2 --seed 42 --no-video

In [ ]:
# Tracked world-model benchmark summary
benchmark_summary = json.loads(Path("demo_assets/world_model_benchmark_summary.json").read_text())
pd.DataFrame(benchmark_summary["world_model_reward_faithfulness"])

## Tiny world-model smoke run

This uses `config/world_model_colab_demo.yaml`, which intentionally reduces model size and writes to isolated demo directories.

The point is executability, not matching the full P5 training budget.

In [ ]:
# Collect a tiny PPO replay dataset for the world-model smoke run
device = "cuda" if torch.cuda.is_available() else "cpu"
!xvfb-run -a python world_model_collect_replay.py --config config/world_model_colab_demo.yaml --policy_checkpoint models/best_model_torch.pt --policy_variant autoresearch_run_008 --train_frames 240 --val_frames 80 --directions CCW --split-prefix colabdemo --device {device}
!python world_model_prepare_dataset.py --config config/world_model_colab_demo.yaml --prefix colabdemo_ppo_ --val-per-manifest 1

In [ ]:
# Train a 1-epoch reduced world model
!python world_model_train.py --config config/world_model_colab_demo.yaml --epochs 1 --save-every 1 --run-name colab_smoke --log-every 5 --batch-log-every 5 --no-wandb

In [ ]:
# Quantitative smoke evaluation
!python scripts/evaluate_reward_faithfulness.py --config config/world_model_colab_demo.yaml --manifest results/world_model/demo_replay/val_manifest.json --world-model-checkpoint models/world_model_demo/rssm_sequence.pt --context-length 10 --horizon 8 --batch-size 4 --num-batches 4 --output results/world_model/demo_artifacts/faithfulness.json

faithfulness = json.loads(Path("results/world_model/demo_artifacts/faithfulness.json").read_text())
pd.DataFrame([
    {
        "reward_mean_mse": faithfulness["reward"]["mean_mse"],
        "reward_mean_corr": faithfulness["reward"]["mean_corr"],
        "speed_mean_corr": faithfulness["telemetry"]["speed"]["mean_corr"],
        "progress_delta_mean_corr": faithfulness["telemetry"]["progress_delta"]["mean_corr"],
        "steer_mean_corr": faithfulness["telemetry"]["steer"]["mean_corr"],
        "offtrack_mean_accuracy": faithfulness["telemetry"]["offtrack"]["mean_accuracy"],
    }
])

In [ ]:
# Generate and display a hallucination comparison video from the first validation episode
from IPython.display import Video

val_manifest = json.loads(Path("results/world_model/demo_replay/val_manifest.json").read_text())
episode_path = Path(val_manifest["episodes"][0])
print("Using episode:", episode_path)

!python scripts/make_demo_video.py --checkpoint models/world_model_demo/rssm_sequence.pt --config config/world_model_colab_demo.yaml --episode {episode_path} --start-index 0 --context-length 10 --horizon 20 --output results/world_model/demo_artifacts/colab_smoke_compare.mp4 --fps 8 --upscale 4

Video("results/world_model/demo_artifacts/colab_smoke_compare.mp4", embed=True)

## Suggested next improvements

To make the Colab story stronger than a smoke run:
- host one small world-model checkpoint and one short replay clip outside git, then add direct download cells
- commit a compact benchmark JSON whenever major world-model experiments finish
- keep demo configs isolated from research configs so notebook runs never overwrite local artifacts
- add a second optional section that evaluates the full P5 checkpoint once hosted